In [ ]:
!pip install mp-api -q

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

category_colors = {
    "Conductor-like": "#FF5722",
    "Semiconductor": "#2196F3",
    "Insulator": "#4CAF50"
}

df_sorted = df_real.sort_values("Bandgap_eV").reset_index(drop=True)
colors = [category_colors[c] for c in df_sorted["Category"]]

plt.figure(figsize=(16, 7))

# Shaded background zones instead of side text
plt.axhspan(0, 1.0, color="#FF5722", alpha=0.08, zorder=0)
plt.axhspan(1.0, 3.0, color="#2196F3", alpha=0.08, zorder=0)
plt.axhspan(3.0, df_sorted["Bandgap_eV"].max() + 0.3, color="#4CAF50", alpha=0.08, zorder=0)

plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.4, linewidth=1)
plt.axhline(y=3.0, color='gray', linestyle='--', alpha=0.4, linewidth=1)

plt.scatter(range(len(df_sorted)), df_sorted["Bandgap_eV"],
            c=colors, s=80, edgecolors='black', linewidth=0.5, zorder=3)

patches = [mpatches.Patch(color=c, label=f"{k} ({'<1.0' if k=='Conductor-like' else '1.0–3.0' if k=='Semiconductor' else '>3.0'} eV)")
           for k, c in category_colors.items()]
plt.legend(handles=patches, loc='upper left', fontsize=10, framealpha=0.95)

plt.title(f"Bandgap Distribution of {len(df_sorted)} Real Materials (Materials Project Database)", fontsize=14)
plt.xlabel("Material Index (sorted by bandgap)", fontsize=12)
plt.ylabel("Bandgap (eV)", fontsize=12)
plt.xlim(-1, len(df_sorted))
plt.tight_layout()
plt.savefig("real_materials_distribution.png", dpi=150)
plt.show()

from google.colab import files
files.download("real_materials_distribution.png")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X = df_real2[["Bandgap_eV"]]
y = df_real2["Is_Metal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_metal2 = DecisionTreeClassifier(max_depth=3)
model_metal2.fit(X_train, y_train)

predictions = model_metal2.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Accuracy: {accuracy * 100:.1f}%")
print()
print(classification_report(y_test, predictions, zero_division=0))

In [ ]:
from mp_api.client import MPRester
import pandas as pd

API_KEY = "T8NM32mVsIRiBobfmd99WiWnIoByjVuX"  # your real key

with MPRester(API_KEY) as mpr:
    docs = mpr.materials.summary.search(
        band_gap=(0, 4.5),
        num_chunks=1,
        chunk_size=150,  # more materials this time
        fields=[
            "material_id", "formula_pretty", "band_gap", "is_metal",
            "density", "volume", "nsites", "energy_above_hull",
            "formation_energy_per_atom"
        ]
    )

data = []
for doc in docs:
    data.append({
        "Material_ID": doc.material_id,
        "Formula": doc.formula_pretty,
        "Bandgap_eV": doc.band_gap,
        "Is_Metal": doc.is_metal,
        "Density": doc.density,
        "Volume": doc.volume,
        "Num_Sites": doc.nsites,
        "Energy_Above_Hull": doc.energy_above_hull,
        "Formation_Energy": doc.formation_energy_per_atom
    })

df_big = pd.DataFrame(data)
print(f"Retrieved {len(df_big)} materials with {df_big.shape[1]} features each")
print()
print(df_big.head())
print()
print("Missing values per column:")
print(df_big.isnull().sum())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# Multiple real features this time, not just bandgap
features = ["Bandgap_eV", "Density", "Volume", "Num_Sites",
            "Energy_Above_Hull", "Formation_Energy"]

X = df_big[features]
y = df_big["Is_Metal"]

print("Is_Metal distribution:")
print(y.value_counts())
print()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Random Forest — a stronger model than a single Decision Tree
model_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_rf.fit(X_train, y_train)

predictions = model_rf.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Random Forest Model — trained on {len(features)} features")
print(f"Training samples: {len(X_train)} | Testing samples: {len(X_test)}")
print(f"Accuracy: {accuracy * 100:.1f}%")
print()
print(classification_report(y_test, predictions, zero_division=0))

# Feature importance — which features mattered most?
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model_rf.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nFeature Importance (what the model relies on most):")
print(importance)

In [ ]:
import matplotlib.pyplot as plt

importance_sorted = importance.sort_values("Importance", ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(importance_sorted["Feature"], importance_sorted["Importance"],
                 color='#4C72B0', edgecolor='black')

for bar, val in zip(bars, importance_sorted["Importance"]):
    plt.text(val + 0.01, bar.get_y() + bar.get_height()/2,
              f'{val:.1%}', va='center', fontsize=10)

plt.xlabel("Importance Score", fontsize=12)
plt.title("Feature Importance — Predicting Metallic Behavior\n(Random Forest, 150 Real Materials)", fontsize=13)
plt.xlim(0, max(importance_sorted["Importance"]) + 0.1)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

from google.colab import files
files.download("feature_importance.png")